# miniLLM v2.2 — évaluations officielles, graphiques, export et publication de **C**

Publie le SFT **C `final.pt`** (pré-entraînement it. 50 200) dans un **nouveau dépôt** : l'ancien dépôt v2.1 n'est pas touché.
Nom du dépôt = même nomenclature que v2.1, calculé à partir du **vrai** checkpoint (itérations, tokens vus, époques) et de la date du jour.

**Deux passages :**
1. **Maintenant (`UPLOAD = False`)** : évaluations officielles sur C, graphiques de loss (pré-entraînement et SFT de C), fiche de chiffres réels, export `safetensors` + test de fumée. **Rien n'est envoyé.**
2. **Plus tard (`UPLOAD = True`)**, une fois le README écrit à partir de la fiche : même notebook, les évaluations déjà faites sont relues (pas refaites), puis création du dépôt et envoi.

Exécution → type d'exécution → **GPU (T4)**, puis **Tout exécuter** (~15 min au 1ᵉʳ passage).

In [ ]:
# ── 1) Réglages, Drive, code ──
UPLOAD = False          # ← passe à True seulement quand README.md et resultats_automatiques.md sont dans le dépôt de code
VERSION = "2.2"
FORCE_EVALS = False     # True = refaire les évaluations même si déjà faites

import os, sys, glob, shutil, json, datetime
from google.colab import drive
drive.mount("/content/drive")

cands = (["/content/drive/MyDrive/MiniLLM_v2"] + glob.glob("/content/drive/MyDrive/*/MiniLLM_v2")
         + glob.glob("/content/drive/Shareddrives/*/MiniLLM_v2"))
BASE = next((c for c in cands if os.path.isdir(f"{c}/data")), None)
if BASE is None:
    raise SystemExit("Dossier MiniLLM_v2 introuvable (Drive > Partagés > appui long > Ajouter un raccourci > Mon Drive).")
DATA, CKPT = f"{BASE}/data", f"{BASE}/checkpoints"
PT_OUT, C_DIR, C_DATA = f"{CKPT}/pretrain", f"{CKPT}/sft_C", f"{DATA}/sft_C"
C_FINAL = f"{C_DIR}/final.pt"
EVAL_DIR = f"{BASE}/evals_v22"
CODE_DIR, OUT = "/content/minillm_v2", "/content/hf_export"

!pip -q install tokenizers datasets huggingface_hub safetensors matplotlib
if os.path.exists(f"{CODE_DIR}/.git"):
    !git -C {CODE_DIR} fetch -q origin v2.1.1 && git -C {CODE_DIR} checkout -q v2.1.1 && git -C {CODE_DIR} pull -q --ff-only
else:
    !git clone -q -b v2.1.1 https://github.com/bono-p/minillm_v2.git {CODE_DIR}
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)

for p in (f"{PT_OUT}/best.pt", C_FINAL, f"{C_DATA}/meta.json", f"{DATA}/tokenizer.json"):
    print("✓" if os.path.exists(p) else "✗ MANQUANT", p)
if not (os.path.exists(C_FINAL) and os.path.exists(f"{PT_OUT}/best.pt")):
    raise SystemExit("Checkpoint manquant : vérifie les ✗ ci-dessus.")

import release_tools as rt
from compare_sft import ensure_tokenizer
ensure_tokenizer(C_FINAL, f"{DATA}/tokenizer.json", C_DATA)


In [ ]:
# ── 2) Évaluations officielles de C (relues si déjà faites) ──
ARGS = {
    "qa":        (["evaluate.py", "qa", "--ckpt", C_FINAL, "--sft_dir", C_DATA, "--n", "500"], True),     # validation PROPRE à C
    "persona":   (["evaluate.py", "persona", "--ckpt", C_FINAL], True),
    "demo":      (["evaluate.py", "demo", "--ckpt", C_FINAL], False),
    "openqa":    (["evaluate.py", "openqa", "--ckpt", C_FINAL, "--n", "1000"], False),
    "basics":    (["basics_eval.py", "--ckpt", C_FINAL], True),
    "skills":    (["skills_eval.py", "--ckpt", C_FINAL, "--n", "60"], True),
}
RAW = {}
for name, (args, parse) in ARGS.items():
    RAW[name], _ = rt.run_eval(args, name, EVAL_DIR, force=FORCE_EVALS, parse_json=parse)
print("\nFichiers dans", EVAL_DIR, ":", sorted(os.listdir(EVAL_DIR)))


In [ ]:
# ── 3) Graphiques de loss (même style que v2.1) + vérification des logs ──
os.makedirs(f"{OUT}/assets", exist_ok=True)
import release_tools as rt
res = {}
for label, log, png in [("pretrain", f"{PT_OUT}/log.jsonl", "loss_pretrain.png"), ("sft", f"{C_DIR}/log.jsonl", "loss_sft.png")]:
    try:
        res[label] = rt.make_loss_plot(log, f"{OUT}/assets/{png}")
        shutil.copy(f"{OUT}/assets/{png}", f"{EVAL_DIR}/{png}")
        print(f"✓ {png} :", {k: v for k, v in res[label].items() if k != "log"})
    except Exception as e:
        print(f"❌ {png} : {e}  (jsonl présents dans le dossier : {glob.glob(os.path.dirname(log) + '/*.jsonl')})")
from IPython.display import Image, display
for png in ("loss_pretrain.png", "loss_sft.png"):
    if os.path.exists(f"{OUT}/assets/{png}"):
        display(Image(f"{OUT}/assets/{png}"))


In [ ]:
# ── 4) Fiche de chiffres RÉELS (à m'envoyer : le README est écrit à partir d'elle) ──
stats = rt.collect_stats(f"{PT_OUT}/best.pt", C_FINAL, C_DATA, f"{PT_OUT}/log.jsonl", f"{C_DIR}/log.jsonl", EVAL_DIR,
                         version=VERSION, date=datetime.date.today())
for k in ("model_config",):
    stats["pretrain"].pop(k, None)
    stats["sft"].pop(k, None)
json.dump(stats, open(f"{EVAL_DIR}/fiche.json", "w"), ensure_ascii=False, indent=1)
REPO_ID = f"bonopassale/{stats['repo_name']}"
print("DÉPÔT :", REPO_ID, "\n")
print(json.dumps(stats, ensure_ascii=False, indent=1))


In [ ]:
# ── 5) Export du modèle + test de fumée (CPU) ──
import export_hf
info = export_hf.export_package(C_FINAL, OUT)
print({k: v for k, v in info.items() if k != "files"})
assert info["max_diff_vs_checkpoint"] == 0, "le fichier exporté diffère du checkpoint !"
RELEASE_DIR = f"{CODE_DIR}/release"
export_hf.assemble_static(OUT, RELEASE_DIR, f"{DATA}/tokenizer.json")
smoke = export_hf.smoke_test(OUT, f"{OUT}/tokenizer.json")
for q, a in smoke.items():
    print("Q:", q, "\nR:", a)
for f in sorted(os.listdir(OUT)):
    p = f"{OUT}/{f}"
    print(f"  {f:28s}", "(dossier)" if os.path.isdir(p) else f"{os.path.getsize(p) / 1e6:8.1f} Mo")


In [ ]:
# ── 6) Publication dans un NOUVEAU dépôt (seulement si UPLOAD = True) ──
if not UPLOAD:
    print("UPLOAD = False : rien n'a été envoyé. Envoie-moi la fiche (cellule 4) et le fichier openqa.txt ; je rédige le README.")
else:
    rel = f"{CODE_DIR}/release/v{VERSION}"
    need = {"README.md": "README.md", "evals/resultats_automatiques.md": "evals/resultats_automatiques.md"}
    missing = [k for k in need if not os.path.exists(f"{rel}/{k}")]
    if missing:
        raise SystemExit(f"Fichiers manquants dans {rel} : {missing} — relance la cellule 1 pour récupérer le dépôt à jour.")
    for src, dst in need.items():
        os.makedirs(os.path.dirname(f"{OUT}/{dst}") or OUT, exist_ok=True)
        shutil.copy(f"{rel}/{src}", f"{OUT}/{dst}")
    open(f"{OUT}/MiniLlmQA.md", "w", encoding="utf-8").write(rt.extract_openqa(open(f"{EVAL_DIR}/openqa.txt", encoding="utf-8").read()))
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass
        HF_TOKEN = getpass("Token Hugging Face (write) : ")
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=OUT, repo_id=REPO_ID, repo_type="model", commit_message=f"miniLLM v{VERSION} : modèle, README, évaluations")
    print("\n✓ Publié :", f"https://huggingface.co/{REPO_ID}")


**À m'envoyer après le 1ᵉʳ passage :** la **fiche** (cellule 4), les sorties de `persona` et `demo` (cellule 2), et le fichier `MiniLLM_v2/evals_v22/openqa.txt` (les 337 réponses de C).
Je rédige alors `README.md` et `evals/resultats_automatiques.md` avec ces chiffres, puis tu passes `UPLOAD = True`.